# 📄 PDF Importer & Exporter (1:1 Exact Match)
This notebook allows you to upload a PDF from your local system and immediately access an exact, unmodified 1:1 copy of it.

In [ ]:
# Install ipywidgets if not already installed
%pip install -q ipywidgets

In [4]:
import base64
import hashlib
from html import escape
from pathlib import Path

import ipywidgets as widgets
from IPython.display import HTML, clear_output, display


def set_status(message, color="#334155"):
    status.value = (
        f"<div style='margin:8px 0; color:{color}; font-family:Arial, sans-serif;'>"
        f"{escape(message)}</div>"
    )


def extract_uploaded_file(upload_value):
    """Return (filename, bytes_content) across ipywidgets versions."""
    if not upload_value:
        return None, None

    if isinstance(upload_value, dict):
        first_name, first_item = next(iter(upload_value.items()))
        raw_content = first_item.get("content", b"")
        return first_name, bytes(raw_content)

    if isinstance(upload_value, tuple):
        first_item = upload_value[0]
        filename = first_item.get("name", "uploaded.pdf")
        raw_content = first_item.get("content", b"")
        return filename, bytes(raw_content)

    return None, None


def render_exact_copy(file_name, file_bytes):
    if not file_name.lower().endswith(".pdf"):
        file_name = f"{file_name}.pdf"

    out_name = f"copy_{file_name}"
    b64_pdf = base64.b64encode(file_bytes).decode("ascii")
    size_kb = len(file_bytes) / 1024
    md5_hash = hashlib.md5(file_bytes).hexdigest()

    set_status(
        f"Loaded {file_name} ({size_kb:.1f} KB). Exact bytes preserved.",
        color="#166534",
    )

    with output_area:
        clear_output()
        display(
            HTML(
                f"""
<div style="margin:16px 0; font-family:Arial, sans-serif;">
  <p style="margin:0 0 10px 0;"><b>File:</b> {escape(file_name)}</p>
  <p style="margin:0 0 14px 0;"><b>MD5:</b> {md5_hash}</p>
  <a href="data:application/pdf;base64,{b64_pdf}" download="{escape(out_name)}"
     style="display:inline-block; padding:12px 22px; background:#0f766e; color:#ffffff;
            text-decoration:none; border-radius:8px; font-weight:700;">
     Download Exact Copy
  </a>
</div>
                """
            )
        )


def on_use_uploaded_pdf(_):
    file_name, file_bytes = extract_uploaded_file(import_button.value)
    if not file_name:
        set_status("No uploaded PDF found. Click 'Import PDF' first.", color="#b45309")
        return

    render_exact_copy(file_name, file_bytes)


def on_load_from_path(_):
    raw_path = path_input.value.strip().strip('"').strip("'")
    if not raw_path:
        set_status("Enter a full PDF path, then click 'Load From Path'.", color="#b45309")
        return

    pdf_path = Path(raw_path).expanduser()
    if not pdf_path.exists() or not pdf_path.is_file():
        set_status("Path not found. Check the file path and try again.", color="#991b1b")
        return

    if pdf_path.suffix.lower() != ".pdf":
        set_status("Selected file is not a PDF.", color="#991b1b")
        return

    try:
        file_bytes = pdf_path.read_bytes()
    except Exception as exc:
        set_status(f"Could not read file: {exc}", color="#991b1b")
        return

    render_exact_copy(pdf_path.name, file_bytes)


def on_browse_system(_):
    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        selected = filedialog.askopenfilename(
            title="Select PDF file",
            filetypes=[("PDF files", "*.pdf")],
        )
        root.destroy()

        if not selected:
            set_status("No file selected.", color="#b45309")
            return

        path_input.value = selected
        on_load_from_path(None)
    except Exception as exc:
        set_status(
            f"System file picker is unavailable: {exc}. Use path input instead.",
            color="#991b1b",
        )


# Top controls
import_button = widgets.FileUpload(
    accept=".pdf",
    multiple=False,
    description="Import PDF",
    button_style="info",
)
use_upload_button = widgets.Button(
    description="Use Uploaded PDF",
    button_style="success",
    icon="check",
)

# Fallback controls
path_input = widgets.Text(
    value="",
    placeholder="C:/Users/YourName/Downloads/file.pdf",
    description="Path:",
    layout=widgets.Layout(width="70%"),
)
browse_button = widgets.Button(description="Browse", icon="folder-open")
load_path_button = widgets.Button(description="Load From Path", button_style="warning")

status = widgets.HTML()
output_area = widgets.Output()

use_upload_button.on_click(on_use_uploaded_pdf)
browse_button.on_click(on_browse_system)
load_path_button.on_click(on_load_from_path)

header = widgets.HTML("<h3 style='margin:0;'>PDF to UI PDF (Exact 1:1 Copy)</h3>")
help_text = widgets.HTML(
    "<p style='margin:6px 0 10px 0;'>"
    "1) Click <b>Import PDF</b> and select your file, then click <b>Use Uploaded PDF</b>.<br>"
    "2) If upload is blocked, click <b>Browse</b> or paste full path and use <b>Load From Path</b>."
    "</p>"
)

ui = widgets.VBox(
    [
        header,
        help_text,
        widgets.HBox([import_button, use_upload_button]),
        widgets.HBox([path_input, browse_button, load_path_button]),
        status,
        output_area,
    ]
)

set_status("Ready. Import a PDF to create an exact copy download.", color="#1d4ed8")
display(ui)